In [1]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
langchain-chroma \
groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9

Mount Google Drive

In [2]:
from google.colab import drive

try:
    drive.mount('/content/drive')
    print("Drive mounted successfully")
except Exception as e:
    print("Error:", e)

Mounted at /content/drive
Drive mounted successfully


In [4]:
import os
from google.colab import userdata

try:

    os.environ["GOOGLE_API_KEY"] = (
        userdata.get("Gemini_key")
    )

    GROQ_API_KEY = userdata.get(
        "Groq_Key"
    )

    print("API Keys Loaded Successfully")

except Exception as e:

    print("Error:", e)

API Keys Loaded Successfully


In [5]:
from langchain_chroma import Chroma

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings
)

from groq import Groq

print("Imported successfully")

Imported successfully


Embedddings

In [6]:
try:

    embeddings = (
        GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001"
        )
    )

    print("Embeddings ready")

except Exception as e:

    print("Error:", e)

Embeddings ready


Connect to Existing ChromaDB

In [7]:
CHROMA_DIR = (
    "/content/drive/MyDrive/"
    "RAG_Internship/chroma_db"
)

try:

    db = Chroma(
        collection_name="wiki_rag",
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR
    )

    print(
        "Documents:",
        db._collection.count()
    )

except Exception as e:

    print("Error:", e)

Documents: 245


Retriver

In [8]:
try:

    retriever = db.as_retriever(
        search_kwargs={"k": 5}
    )

    print("Retriever ready")

except Exception as e:

    print("Error:", e)

Retriever ready


In [9]:
query = "What is machine learning?"

results = retriever.invoke(query)

print(
    f"Retrieved {len(results)} chunks"
)

Retrieved 5 chunks


In [10]:
for i, doc in enumerate(results, 1):

    print("=" * 50)

    print(f"Chunk {i}")

    print(
        doc.page_content[:500]
    )

Chunk 1
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from  data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Statistics and mathematical optimisation methods compose the foundations o
Chunk 2
Models
A machine learning model is a type of mathematical model that, once "trained" on a given dataset, can be used to make predictions or classifications on new data. During training, a learning algorithm iteratively adjusts the model's internal parameters to minimise errors in its predictions. By extension, the term "model" can refer to several levels of specificity, from a general class of models and their associated learning algorithms to a fully trained model with all its 

Answer Generation

In [12]:
try:

    client = Groq(
        api_key=GROQ_API_KEY
    )

    print("Groq initialized Successfully")

except Exception as e:

    print("Error:", e)

Groq initialized Successfully


Create RAG Function

In [13]:
def ask_question(query):

    try:

        docs = retriever.invoke(query)

        if len(docs) == 0:

            return (
                "No relevant documents found."
            )

        context = "\n\n".join(
            [
                doc.page_content
                for doc in docs
            ]
        )

        prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is not present
in the context, say:

'I could not find that information
in the documents.'

Context:
{context}

Question:
{query}
"""

        response = (
            client.chat.completions.create(
                model=
                "llama-3.3-70b-versatile",
                messages=[
                    {
                        "role":"user",
                        "content":prompt
                    }
                ]
            )
        )

        return (
            response
            .choices[0]
            .message
            .content
        )

    except Exception as e:

        return f"Error: {e}"

Queries

In [14]:
print(
    ask_question(
        "What is deep learning?"
    )
)

Deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is used to transform input data into a progressively more abstract and composite representation. It is based on multi-layered neural networks and can learn which features to optimally place at which level on its own, without the need for hand-crafted feature engineering.


In [15]:
print(
    ask_question(
        "What is computer vision?"
    )
)

Computer vision is an interdisciplinary field that deals with how computers can be made to gain high-level understanding from digital images or videos. It seeks to automate tasks that the human visual system can do, and involves the development of a theoretical and algorithmic basis to achieve automatic visual understanding.


In [16]:
print(
    ask_question(
        "Who is Alan Turing?"
    )
)

Alan Turing is a mathematician and philosopher who investigated whether machines can show intelligent behaviour and think. He proposed the Turing test in 1950, which measures the ability of a machine to simulate human conversation.
